# 16d — the systematic finder: where does OUR fine-tuning emit SDK-illegal answers?

**Where this came from.** 16a asked an off-template question and the fine-tuned checkpoint
answered `"1."` — trailing period — on **87%** of `number` questions. `Number.verify` gates on
`str.strip().isdigit()`, so all of those are **auto-incorrect**. Base: **0.0000**. We created it.
That was luck. This notebook makes it a sweep.

🟢 **It needs no ground truth.** "Would the SDK's verifier accept this string?" is a property of
the model's OUTPUT alone, so any phrasing can be audited without annotating anything.

🔴 **And that is why the defect was invisible.** Our scored eval only ever asks the corpus's own
templates, so a fragility living outside them cannot appear in any number we report — by
construction. The hidden test is the organizers' generator, not ours.

**Design.** Real corpus questions, asked under meaning-preserving surface variants (`v0` is
verbatim = the control). Per (model × format × variant): the illegal-answer rate and *which*
violation occurred. The product is `regressions()` — the list of places where `ft` is more
illegal than `base`, or more illegal than its own `v0`.

⚠️ **This measures format legality, not correctness.** Every illegal answer is scored wrong
regardless of whether the model knew the answer, so the rate is a **floor** on lost points.


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import logging, os, sys, json
from pathlib import Path

REPO = Path("/workspace/repo")
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "vendor" / "orena-focus" / "src"))
sys.path.insert(0, str(Path.cwd() / "_models"))          # experiment-private glue

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# --- parameters (RAW LITERALS ONLY; papermill injects overrides directly below) --
SMOKE = True
SEED  = 0

RUN        = "16d_format_audit_v1"
DATA_ROOT  = "/workspace/orena-data"
MODEL_BASE = "/workspace/models/qwen3-vl-8b"
# ep3 = the current best (bucket_mean 0.5724)
MODEL_FT   = "/workspace/repo/experiments/06-vit-lora/runs/06_vit_lora_v1/merged/checkpoint-2580"
# 🔴 ep2 = THE CHECKPOINT ACTUALLY SHIPPED IN SUBMISSION 01 (checkpoint-1720, step 1720).
# 16a probed ep3 only, so the shipped artifact has never been audited for this defect. The
# container does NOT normalise: `answer_one` does `.strip()` and nothing else
# (`experiments/06-vit-lora/_tools/submission/inference.py:336`), so a "1." goes straight to
# Number.verify and is auto-incorrect. This arm is the one that answers "did we already ship it?"
MODEL_EP2  = "/workspace/repo/experiments/06-vit-lora/runs/06_vit_lora_v1/merged/checkpoint-1720"

FORMATS      = ("number", "binary", "fo_class")   # the exact-match formats; judge formats have no gate to mirror
N_PER_FORMAT_SMOKE = 6
N_PER_FORMAT_FULL  = 40
MAX_NEW_TOKENS     = 32


In [ ]:
# --- derived (MUST live below the parameters cell) -------------------------------
DATA_ROOT = Path(DATA_ROOT)
RUN_DIR   = Path.cwd() / "runs" / RUN
# order matters only for readability; "ep2_shipped" is the arm that answers the submission question
MODELS    = {"base": Path(MODEL_BASE), "ep2_shipped": Path(MODEL_EP2), "ft": Path(MODEL_FT)}
N_PER_FORMAT = N_PER_FORMAT_SMOKE if SMOKE else N_PER_FORMAT_FULL
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run dir:", RUN_DIR, "| SMOKE:", SMOKE, "| n/format:", N_PER_FORMAT)


In [ ]:
# --- select real corpus questions per format (gold not needed — see the header) ---
from frame.config import BaselineConfig
from frame.data import load_frame_items
import format_audit as fa

items = load_frame_items(BaselineConfig(data_root=DATA_ROOT), splits=("test",))
items_by_fmt = {f: fa.select_by_format(items, f, N_PER_FORMAT, seed=SEED) for f in FORMATS}

for f, v in items_by_fmt.items():
    assert v, f"no items selected for format {f!r}"
    print(f"{f:10s} n={len(v):3d} videos={len({i.video_id for i in v}):3d}")
n_calls = sum(len(v) for v in items_by_fmt.values()) * len(fa.VARIANTS) * len(MODELS)
print("variants:", list(fa.VARIANTS), "| total model calls:", n_calls)

# show the variants on one real question so the transformation is reviewable, not implicit
_q = str(items_by_fmt["number"][0].request.question)
for name, fn in fa.VARIANTS.items():
    print(f"  {name:14s} {fn(_q)[:90]}")


In [ ]:
# --- run both models over every (question x variant) -----------------------------
import gc, time
from frame.engine import QwenFrameEngine

rows = []
for tag, path in MODELS.items():
    t0 = time.time()
    eng = QwenFrameEngine(BaselineConfig(data_root=DATA_ROOT, model_path=path,
                                         max_new_tokens=MAX_NEW_TOKENS))
    eng.load()
    rows += fa.run_audit(eng, items_by_fmt, model_tag=tag)
    print(f"{tag}: done in {time.time()-t0:.0f}s")
    del eng; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# --- the product: where fine-tuning made us MORE illegal --------------------------
import pandas as pd, json

df = pd.DataFrame(rows); df.to_csv(RUN_DIR / "rows.csv", index=False)
res = fa.score(rows); (RUN_DIR / "score.json").write_text(json.dumps(res, indent=2))

tbl = pd.DataFrame(res).T[["n", "illegal_rate", "top_violation"]]
tbl.index = pd.MultiIndex.from_tuples([tuple(k.split("|")) for k in tbl.index],
                                      names=["model", "fmt", "variant"])
print("=== illegal-answer rate (lower is better; this is a FLOOR on lost points) ===")
print(tbl.unstack("model")["illegal_rate"].round(4).to_string())
print()
print("=== REGRESSIONS: a fine-tuned model more illegal than base, or than its own v0 ===")
regs = fa.regressions(res)
if regs:
    print(pd.DataFrame(regs).to_string(index=False))
    pd.DataFrame(regs).to_csv(RUN_DIR / "regressions.csv", index=False)
else:
    print("none above the 0.10 threshold — a faithful negative, and worth recording as one")
print()
print("=== violation histogram, per model ===")
print(df[~df.sdk_legal].groupby(["model", "fmt", "violation"]).size().to_string())

# 🔴 The question this arm exists to answer.
print()
print("=== DID SUBMISSION 01 SHIP THIS DEFECT? (ep2_shipped = checkpoint-1720) ===")
sub = df[(df.model == "ep2_shipped") & (df.fmt == "number")]
off = sub[sub.variant != "v0_verbatim"]
v0  = sub[sub.variant == "v0_verbatim"]
print(f"  ep2 illegal rate, corpus phrasing (v0) : {1 - v0.sdk_legal.mean():.4f}  n={len(v0)}")
print(f"  ep2 illegal rate, off-template phrasing: {1 - off.sdk_legal.mean():.4f}  n={len(off)}")
print(f"  ep3 illegal rate, off-template phrasing: "
      f"{1 - df[(df.model=='ft') & (df.fmt=='number') & (df.variant!='v0_verbatim')].sdk_legal.mean():.4f}")
print("  NOTE: the shipped container does NOT normalise -- inference.py:336 is .strip() only,")
print("  so any illegal string reaches Number.verify unchanged and is scored wrong.")


In [ ]:
# --- the fixable list: one real example per (fmt, violation) ----------------------
bad = df[(df.model == "ft") & (~df.sdk_legal)]
for (f, v), g in bad.groupby(["fmt", "violation"]):
    r = g.iloc[0]
    print(f"[{f} / {v}]  n={len(g)}")
    print(f"    Q: {r.question[:100]}")
    print(f"    A: {r.raw!r}")
